<br>

<img src="https://lindas.admin.ch/lindaslogo2.png" style="width:25%; float:right">

# LINDAS Tutorial

This page serves as a introductory tutorial to the LINDAS ecosystem. LINDAS is the Linked Data ecosystem of the Swiss Federal Administration.



# Introduction

The webpage you are currently viewing is a so called **interactive JupyterLite notebook**. In this notebook, you can change interactively the content of the single cells and execute these cells directly seeing the result of your changes immediately. The cells contain either [Markdown](https://en.wikipedia.org/wiki/Markdown) content (like this cell) or executable Python source code.

JupyterLite stems from JupyterLab with the advantage of being completely browser based without any backend infrastructure. This means that the execution of the cells could take some time during first execution. Subsequent executions will be much faster because of stored data in your browser cache.

If you are unfamiliar with the handling of Jupyter notebooks, here are two useful resources:

- [The JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [The Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

# Preliminaries

In this part some preliminaries for querying LINDAS with SPARQL are introduced. If you are only interested in the actual LINDAS tutorial, you can skip the whole section and start [here](#Actual-Tutorial).

## Install and Import the Necessary Modules

Querying a SPARQL endpoint is basically making a POST request to the corresponding endpoint URL. As JupyterLite at the moment has no support for Python's `requests` modul, the JavaScript fetch API is used (with some tricks). To make this happen, the following modules have to be importet: 

In [1]:
%pip install -q folium

In [17]:
import json
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO

## Define Main Query Function

As the JavaScript fetch API is asynchronous, the corresponding Python function `query` has to be declared as `async`. This function allows to query the 3 Swiss governmental triple stores.

In [3]:
async def query(query_string, store = "L"):
    
    # three Swiss triplestores
    if store == "F":
        address = 'https://fedlex.data.admin.ch/sparqlendpoint'
    elif store == "G":
        address = 'https://geo.ld.admin.ch/query'
    else:
        address = 'https://ld.admin.ch/query'
    
    # try the Post request with help of JS fetch
    # the creation of the request header is a little bit complicated because it needs to be a 
    # JavaScript JSON that is made within a Python source code
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "text/csv" })),
        )
    except:
        raise RuntimeError("fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        # ld.admin.ch throws errors starting with '{"message":'
        if '{"message":' in res:
            error = json.loads(res)
            raise RuntimeError("SPARQL query malformed: " + error["message"])
        # geo.ld.admin.ch throws errors starting with 'Parse error:'
        elif 'Parse error:' in res:
            raise RuntimeError("SPARQL query malformed: " + res)
        else:
            # if everything works out, create a pandas dataframe from the csv result
            df = pd.read_csv(StringIO(res))
            return df
    else:
        # fedlex.data.admin.ch throws error with response status 400
        if resp.status == 400:
            raise RuntimeError("Response status 400: Possible malformed SPARQL query. No syntactic advice available.")
        else:
            raise RuntimeError("Response status " + resp.status)

If you are interested in the details of using the JavaScript fetch API within JupyterLite, please consult:

- https://pyodide.org/en/stable/usage/faq.html#how-can-i-use-fetch-with-optional-arguments-from-python
- https://github.com/jupyterlite/jupyterlite/discussions/412
- https://lwebapp.com/en/post/pyodide-fetch

## Define Display Function

Displays pandas dataframe resulting from the SPARQL query as HTML with clickable links.

In [4]:
def display_result(df):
    df = HTML(df.to_html(render_links=True, escape=False))
    display(df)

# Actual Tutorial

## All Datasets

One of the basic elements in LINDAS are datasets that group similar data. The first task is to query LINDAS for all available datasets. Datasets in LINDAS have the [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [schema:Dataset](https://schema.org/Dataset).

In [5]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name.
        OPTIONAL {
            ?dataset schema:version ?version.
        }
        
    FILTER(lang(?name) = 'de')
}
ORDER BY (?dataset)

""")

display_result(df)

,dataset,name,version
0,https://agriculture.ld.admin.ch/foen/animal-pest/dataset,Tierseuchen,NaN
1,https://culture.ld.admin.ch/.well-known/dataset/ais,Archivdatenbank des Schweizerischen Bundesarchiv,NaN
2,https://culture.ld.admin.ch/.well-known/dataset/isil,ISIL-Kennzeichen,NaN
3,https://culture.ld.admin.ch/sfa/StateAccounts_Category/1/,Staatsrechnungen - Kategorie,1.0
4,https://culture.ld.admin.ch/sfa/StateAccounts_Category/2/,Staatsrechnungen - Kategorie,2.0
5,https://culture.ld.admin.ch/sfa/StateAccounts_Category/3/,Staatsrechnungen - Kategorie,3.0
6,https://culture.ld.admin.ch/sfa/StateAccounts_Category/4/,Staatsrechnungen - Kategorie,4.0
7,https://culture.ld.admin.ch/sfa/StateAccounts_Category/5/,Staatsrechnungen - Kategorie,5.0
8,https://culture.ld.admin.ch/sfa/StateAccounts_Category/6,Staatsrechnungen - Kategorie,6.0
9,https://culture.ld.admin.ch/sfa/StateAccounts_Domain/1/,Staatsrechnungen - Bereich,1.0


As you can see, there are multiple datasets, that exist in different versions. If you click on e.g. https://environment.ld.admin.ch/foen/ubd000504/3, you will get a useful webpage with additional infos on this dataset. For example, the [purl:creator](http://purl.org/dc/terms/creator) https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu is stated.

## Datasets from a Specific Creator

We can use this [purl:creator](http://purl.org/dc/terms/creator) value to refine our query for showing only datasets from the BAFU:

In [6]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name;
        purl:creator <https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu>.
        
    FILTER(lang(?name) = 'de')
}

""")

display_result(df)

,dataset,name
0,https://environment.ld.admin.ch/foen/ubd0037/1/,Lärmbelastung durch Verkehr
1,https://environment.ld.admin.ch/foen/ubd0104/1/,Qualität der Badegewässer
2,https://environment.ld.admin.ch/foen/ubd0066/1/,Schwermetallbelastung des Bodens
3,https://environment.ld.admin.ch/foen/ubd0066/2/,Schwermetallbelastung des Bodens
4,https://environment.ld.admin.ch/foen/ubd0037/2/,Lärmbelastung durch Verkehr
5,https://environment.ld.admin.ch/foen/ubd0066/3/,Schwermetallbelastung des Bodens
6,https://environment.ld.admin.ch/foen/ubd0104/2/,Qualität der Badegewässer
7,https://environment.ld.admin.ch/foen/ubd0037/3/,Lärmbelastung durch Verkehr
8,https://environment.ld.admin.ch/foen/ubd0066/4/,Schwermetallbelastung des Bodens
9,https://environment.ld.admin.ch/foen/ubd0104/3/,Qualität der Badegewässer


## Working with a specific Dataset (Bathing water quality)

Assume that we would be intersted in the Bathing water quality dataset. Our first task is to get the latest version of the dataset. If we click on a arbitrary version of the dataset (e.g. https://environment.ld.admin.ch/foen/ubd0104/3/), we see that the datasat has a [purl:identifier](http://purl.org/dc/terms/identifier) that reads "ubd0104". It turns out, that this identifier ist stable through ther versions, so we can query for the latest version with the help of a SPARQL [subquery](https://en.wikibooks.org/wiki/SPARQL/Subqueries) construction:

In [7]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT * WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?maxversion.

    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

""")

display_result(df)

,dataset,maxversion
0,https://environment.ld.admin.ch/foen/ubd0104/6,6


As this dataset is a [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [cube:Cube](https://cube.link/#Cube), the actual measurements are organised within a [cube:ObservationSet](https://cube.link/#ObservationSet):

In [8]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT ?observation WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 10

""")

display_result(df)

,observation
0,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-05-23/E.coli/CH10001
1,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-05-26/E.coli/CH10001
2,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-05-23/Enterokokken/CH10001
3,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-08-13/Enterokokken/CH10001
4,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-08-11/E.coli/CH10001
5,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-07-24/E.coli/CH10001
6,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2018-09-03/E.coli/CH10001
7,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-06-23/Enterokokken/CH10001
8,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-05-26/Enterokokken/CH10001
9,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-08-11/Enterokokken/CH10001


If we want to have tabular data, we can observe a single observation and see the used vocabulary for creating the following query:

In [9]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?dateOfProbing ?parameterType ?value WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:dateofprobing ?dateOfProbing;
        ubd0104:station ?station;
        ubd0104:parametertype ?parameterType;
        ubd0104:station ?station;
        ubd0104:value ?value.
    
    ?station schema:name ?name.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 100

""")

display_result(df)

,name,station,dateOfProbing,parameterType,value
0,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-05-23,E.coli,210
1,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-05-26,E.coli,80
2,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-05-23,Enterokokken,62
3,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-08-13,Enterokokken,31
4,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-08-11,E.coli,21
5,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2019-07-24,E.coli,17
6,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2018-09-03,E.coli,15
7,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-06-23,Enterokokken,14
8,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-05-26,Enterokokken,14
9,Plage de Gumefens,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,2020-08-11,Enterokokken,12


Our next goal is to show all the available stations on a map of Switzerland. The SPARQL query is:

In [10]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>


SELECT DISTINCT ?station ?canton ?name ?latitude ?longitude WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:station ?station.
    ?station ubd0104:kanton ?canton;
        schema:name ?name;
        schema:latitude ?latitude;
        schema:longitude ?longitude.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

""")

display_result(df)

,station,canton,name,latitude,longitude
0,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,FR,Plage de Gumefens,46.6753,7.0879
1,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10004,FR,Gemeinde Strandbad,46.9279,7.1109
2,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10007,FR,Plage de Portalban,46.9227,6.9534
3,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10009,FR,Nouvelle plage,46.8571,6.8483
4,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1013,ZH,Strandbad Maur,47.3456,8.6747
5,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1016,ZH,Strandbad Uster,47.3414,8.6911
6,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1029,ZH,Strandbad Baumen,47.3583,8.7855
7,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1050,ZH,Strandbad Küsnacht,47.3093,8.5826
8,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1058,ZH,Seebad Lattenberg,47.2390,8.7171
9,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1061,ZH,Strandbad Bürger,47.2891,8.5749


To display all the stations on a map, we use the Python [folium](https://python-visualization.github.io/folium/) module:

In [11]:
# create the map and center it around the mean coordinates of the stations
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="stamenterrain", zoom_start=8)

# add marker one by one on the map
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa')
   ).add_to(m)

# show the map
m

To improve the map, we want to display the marker in different colors corresponding to the quality of the last measurements and by clicking on the marker, a popup with additional information should appear.

In [12]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?latitude ?longitude ?date ?value WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date;
            ubd0104:value ?value.
        
        ?station schema:name ?name;
            schema:latitude ?latitude;
            schema:longitude ?longitude.
    

    {
    SELECT ?station (max(?date) as ?date) WHERE {

        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date.
    }

    GROUP BY ?station
    }
}
""")

display_result(df)

,name,station,latitude,longitude,date,value
0,Genève-plage,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH25016,46.2144,6.1734,2021-09-06,57
1,Aare Lorraine,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH24002,46.9579,7.4419,2021-09-27,55
2,Rhône Pont-sous-Terre RD,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH25034,46.2036,6.1307,2021-09-06,51
3,"Strandbad Stampf, Zürichsee",https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH17028,47.2159,8.8430,2021-08-10,100
4,Bain des Dames,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH22011,46.4892,6.7366,2021-08-10,100
5,Gland/La Falaise,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH22016,46.4186,6.2901,2021-08-19,100
6,Curtinaux,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH22029,46.4989,6.6922,2021-08-16,100
7,Meriggio,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH21094,46.1797,8.7592,2021-09-06,30
8,Camping,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH22038,46.3948,6.8965,2021-08-19,100
9,Zona Coop,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH21099,46.2437,8.7044,2021-09-06,30


Now we want to create the map.

In [13]:
# create column with quality colors
conditions = [(df['value'] < 10),
              (df['value'] >= 10) & (df['value'] < 50),
              (df['value'] >= 50) & (df['value'] < 100),
              (df['value'] >= 100)
             ]

values = ["green", "darkgreen", "orange", "red"]

df['quality'] = np.select(conditions, values)

# create the map and center it around the mean coordinates of the stations
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="stamenterrain", zoom_start=8)

# add marker one by one on the map
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<p><a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a></p>' + 
                                       '<p>Date: ' + df.iloc[i]['date'] + '</p>' + 
                                       '<p>Value E.Coli = ' + str(df.iloc[i]["value"]) + '</p>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa', color = df.iloc[i]['quality'])
   ).add_to(m)

# show the map
m

## Defined Terms

In [14]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT * WHERE {
    
    ?DefinedTermSet a schema:DefinedTermSet;
        schema:name ?Name.
    FILTER(regex(str(?DefinedTermSet), "admin.ch" ) )
    FILTER(lang(?Name) = "de")
}

""")

display_result(df)

,DefinedTermSet,Name
0,https://culture.ld.admin.ch/isil,ISIL-Kennzeichen
1,https://register.ld.admin.ch/opendataswiss/category,Kategorien von opendata.swiss
2,https://ld.admin.ch/ech/97/legalforms,Rechtsformen
3,https://ld.admin.ch/dimension/zefix,Zefix - Zentraler Firmenindex
4,https://ld.admin.ch/dimension/department,Eidgenössische Departemente und die Bundeskanzlei
5,https://ld.admin.ch/dimension/office,Bundesämter
6,https://ld.admin.ch/dimension/canton,Kantone
7,https://ld.admin.ch/dimension/municipality,Gemeinden
8,https://ld.admin.ch/cube/dimension/el01,Schwermetall
9,https://register.ld.admin.ch/opendataswiss/org,Die Organisationen von opendata.swiss


## geo.ld.admin.ch

In [15]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?Municipality ?Name ?Population WHERE {
    ?Municipality gn:featureCode gn:A.ADM3 .
    ?Municipality schema:name ?Name .
    ?Municipality gn:population ?Population .
    ?Municipality <http://purl.org/dc/terms/issued> ?Date .
    FILTER (?Date = "2022-01-01"^^xsd:date)
}
ORDER BY DESC(?Population)
LIMIT 5

""", "G")

display_result(df)

,Municipality,Name,Population
0,https://geo.ld.admin.ch/boundaries/municipality/261:2022,Zürich,421878
1,https://geo.ld.admin.ch/boundaries/municipality/6621:2022,Genève,203856
2,https://geo.ld.admin.ch/boundaries/municipality/2701:2022,Basel,173863
3,https://geo.ld.admin.ch/boundaries/municipality/5586:2022,Lausanne,140202
4,https://geo.ld.admin.ch/boundaries/municipality/351:2022,Bern,134794


## Choropleth Maps

In [74]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?bfs ?population ?geometry WHERE {

?muni gn:featureCode gn:A.ADM3;
    schema:about ?about;
    <https://geo.ld.admin.ch/def/bfsNumber> ?bfs;
    <http://purl.org/dc/terms/hasVersion> ?version.
    
?version <http://purl.org/dc/terms/issued> "2023-01-01"^^xsd:date;
    <http://www.opengis.net/ont/geosparql#defaultGeometry>/<http://www.opengis.net/ont/geosparql#asWKT> ?geometry;
    <http://www.geonames.org/ontology#population> ?population.
    
    
FILTER(?bfs < 400)
FILTER(?bfs > 350)
    
}

""", "G")

df = gpd.GeoDataFrame(df)

df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")

m = folium.Map(location=[5.170035, -74.914305], tiles='cartodbpositron', zoom_start=6)
folium.Choropleth(
    geo_data=df,
    name="choropleth",
    columns=["population"]
).add_to(m)
folium.LayerControl().add_to(m)
display(m)


In [53]:
df.head()

,bfs,population,geometry
0,351,134290,"POLYGON ((7.33122751567709 46.9192346180716,7...."
1,352,6404,"POLYGON ((7.53595837953759 47.0030958997107,7...."
2,353,4340,"POLYGON ((7.42817300002168 46.9738048299477,7...."
3,354,3229,"POLYGON ((7.43960150432261 47.0077480549072,7...."
4,355,42177,"POLYGON ((7.33122751567709 46.9192346180716,7...."


In [22]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT * WHERE {

<https://geo.ld.admin.ch/boundaries/municipality/geometry-g1/351:2022> <http://www.opengis.net/ont/geosparql#asWKT> ?wkt.

}

""", "G")

s = gpd.GeoSeries.from_wkt(df["wkt"])

display(s)

0    POLYGON ((7.29516 46.92253, 7.29427 46.92423, ...
dtype: geometry

## fedlex.data.admin.ch

In [16]:
df = await query("""

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?eventLabel ?decisionDate ?sourceId  WHERE {
  ?actAbstract jolux:classifiedByTaxonomyEntry ?concept .
  ?concept skos:notation ?notation .
  filter(datatype(?notation) = <https://fedlex.data.admin.ch/vocabulary/notation-type/id-systematique>)
  filter(str(?notation) = '170.512')

  ?actAbstract  jolux:basicAct ?basicAct .
  ?draft jolux:hasResultingLegalResource ?basicAct .
  ?draft rdf:type jolux:InitialDraft .

  ?draft jolux:draftHasLegislativeTask ?event . 
  ?event jolux:legislativeTaskType ?eventType .
  ?eventType skos:prefLabel ?eventLabel . filter(lang(?eventLabel) = 'fr')
  ?event jolux:decisionDate ?decisionDate .
  optional {
    ?event jolux:legislativeTaskHasResultingLegalResource ?source .
    ?source jolux:isRealizedBy ?expression .
    ?expression jolux:historicalLegalId ?sourceId .
    ?expression jolux:language <http://publications.europa.eu/resource/authority/language/FRA> .
  }
} 
order by desc(?decisionDate)

""", "F")

display_result(df)

,eventLabel,decisionDate,sourceId
0,Expiration du délai référendaire,2004-10-07,NaN
1,Arrêté du Parlement,2004-06-18,FF 2004 2919
2,Message du Conseil fédéral,2003-10-22,FF 2003 7047
3,Arrêté du Parlement,1986-03-21,FF 1986 I 858
4,Message du Conseil fédéral,1983-06-29,FF 1983 III 441
